# The bradyzoite axis at the protein level


**The open point this tests (instruction 75).** The bradyzoite fingerprint is built on RNA. Does it read a proteome? Until this snapshot no finished proteomics project stated a bradyzoite sample; PXD058095 does.

**The comparison.** PXD058095 has no internal tachyzoite control, so the controls are external: the finished whole-proteome projects of tachyzoites from the same uniform Sage search. Scores are within-sample ranks, so instrument and loading cannot carry the answer.

**What would count as a pass.** The stated bradyzoite proteome scores above every tachyzoite proteome. Anything less is reported as it falls.

Built 2026-11-08. The NAS is read-only here.

In [1]:
import glob, json, os, numpy as np, pandas as pd
fp = pd.read_csv(os.path.join(REPO, 'results/condition_fingerprints_2026_11_06/fingerprints.tsv'), sep='\t')
brady = fp[fp.fingerprint == 'bradyzoite_conversion_gse223869']
rises = set(brady.loc[brady.set == 'rises', 'gene'])
falls = set(brady.loc[brady.set == 'falls', 'gene'])
classification = pd.read_csv(os.path.join(REPO, 'results/pulldown_background_2026_11_08/project_classification.tsv'), sep='\t')
whole = classification[classification['class'] == 'whole_proteome']
print(len(rises), 'genes rise on conversion;', len(falls), 'fall')
print(len(whole), 'finished whole-proteome projects in the uniform search')
whole[['project', 'title']].head(20)

150 genes rise on conversion; 150 fall
15 finished whole-proteome projects in the uniform search


,project,title
79,PXD000261,Host response profile of human brain proteome in toxoplasma encephalitis
80,PXD001042,Proteogenomic analysis of Toxoplasma gondii (comparing genome releases 6 and 10) to demonstrate the performance of the ProteoAnnotator software
81,PXD004895,PROTEOMIC PROFILING OF EXTRACELLULAR VESICLES OBTAINED FROM Toxoplasma gondii AND TOXOPLASMA-INFECTED CELLS
82,PXD008448,Proteomic profiling of human decidual immune proteins during Toxoplasma gondii infection
83,PXD021736,Differential proteomic analysis in a host–parasite interaction of host cells infected with Toxoplasma gondii
84,PXD021938,Mass spectrometry of circulating antigens products isolated from sera of mice infected with Toxoplasma gondii tachyzoites
85,PXD035658,Prioritization of antimicrobial targets by CRISPR-based oligo recombineering
86,PXD040598,Protein abundances following 24h of SPARK depletion in Toxoplasma gondii
87,PXD040635,SPARKEL depletion time course proteome
88,PXD044588,Apical annuli are specialised sites of post-invasion secretion of dense granules in Toxoplasma


## 1 · One score per project

Each project LFQ table is read, protein intensities are summed per gene over its runs, and the genes are ranked within the project. The score is the mean rank of the genes that rise on conversion minus the mean rank of those that fall, in units of the rank range: positive means the proteome looks converted. A project measuring fewer than 40 of either set is reported as too thin, not scored.

In [2]:
MIN_GENES = 40
def gene_ranks(project):
    paths = sorted(glob.glob(os.path.join(MS_ROOT, project, 'sage_*', 'lfq.tsv')))
    frames = []
    for path in paths:
        table = pd.read_csv(path, sep='\t')
        protein = next((c for c in table.columns if c.lower() in ('proteins', 'protein', 'proteinids')), None)
        value_columns = [c for c in table.columns if table[c].dtype.kind == 'f' and c.lower() not in ('q_value', 'spectral_angle', 'score')]
        if protein is None or not value_columns:
            continue
        genes = table[protein].astype(str).str.extract(r'(TGME49_\d+)')[0]
        frames.append(pd.DataFrame({'gene': genes, 'value': table[value_columns].sum(axis=1)}).dropna())
    if not frames:
        return None
    joined = pd.concat(frames).groupby('gene').value.sum()
    joined = joined[joined > 0]
    return joined.rank(pct=True) if len(joined) else None
rows = []
for project in ['PXD058095'] + [p for p in whole.project if p != 'PXD058095']:
    ranks = gene_ranks(project)
    title = classification.loc[classification.project == project, 'title']
    title = str(title.iloc[0]) if len(title) else ''
    if ranks is None:
        rows.append(dict(project=project, title=title[:90], verdict='no readable LFQ table'))
        continue
    up, down = ranks.reindex(sorted(rises)).dropna(), ranks.reindex(sorted(falls)).dropna()
    if len(up) < MIN_GENES or len(down) < MIN_GENES:
        rows.append(dict(project=project, title=title[:90], genes=len(ranks), rises_measured=len(up),
                         falls_measured=len(down), verdict='too few fingerprint genes measured'))
        continue
    rows.append(dict(project=project, title=title[:90], genes=len(ranks), rises_measured=len(up),
                     falls_measured=len(down), score=round(float(up.mean() - down.mean()), 4),
                     verdict='scored'))
S = pd.DataFrame(rows)
S.to_csv(os.path.join(OUT, 'project_scores.tsv'), sep='\t', index=False)
print(S.verdict.value_counts().to_dict())
cols = [c for c in ('project', 'score', 'rises_measured', 'falls_measured', 'title') if c in S.columns]
S[S.verdict == 'scored'].sort_values('score', ascending=False)[cols] if 'score' in S.columns else S[cols]

{'too few fingerprint genes measured': 11, 'no readable LFQ table': 4}


,project,rises_measured,falls_measured,title
0,PXD058095,10,36,Identification of immunogenic markers in Toxoplasma gondii bradyzoite proteins
1,PXD000261,0,1,Host response profile of human brain proteome in toxoplasma encephalitis
2,PXD001042,6,52,Proteogenomic analysis of Toxoplasma gondii (comparing genome releases 6 and 10) to demons
3,PXD004895,0,47,PROTEOMIC PROFILING OF EXTRACELLULAR VESICLES OBTAINED FROM Toxoplasma gondii AND TOXOPLAS
4,PXD008448,0,0,Proteomic profiling of human decidual immune proteins during Toxoplasma gondii infection
5,PXD021736,NaN,NaN,Differential proteomic analysis in a host–parasite interaction of host cells infected with
6,PXD021938,NaN,NaN,Mass spectrometry of circulating antigens products isolated from sera of mice infected wit
7,PXD035658,NaN,NaN,Prioritization of antimicrobial targets by CRISPR-based oligo recombineering
8,PXD040598,3,54,Protein abundances following 24h of SPARK depletion in Toxoplasma gondii
9,PXD040635,5,55,SPARKEL depletion time course proteome


## 2 · Does the stated bradyzoite proteome stand out?

Its rank among the scored projects, and the gap to the next one. With one stated bradyzoite sample this is a single observation, not a validated axis: it can fail the axis, or support it, and cannot establish it. If no project measures enough of the fingerprint, that is the result: the depth of the proteomes, not the fingerprint, is what blocks the axis.

In [3]:
has = 'score' in S.columns and (S.verdict == 'scored').any()
scored = (S[S.verdict == 'scored'].sort_values('score', ascending=False).reset_index(drop=True)
          if has else pd.DataFrame(columns=['project', 'score', 'title']))
rank = (int(scored.index[scored.project == 'PXD058095'][0]) + 1
        if len(scored) and (scored.project == 'PXD058095').any() else None)
if not len(scored):
    verdict = ('no project measures enough of the fingerprint: the depth of these proteomes, not the '
               'fingerprint, is what blocks the axis')
elif rank == 1:
    verdict = 'the stated bradyzoite proteome scores highest'
elif rank is None:
    verdict = 'the stated bradyzoite proteome could not be scored'
else:
    verdict = f'the stated bradyzoite proteome ranks {rank} of {len(scored)} -- the fingerprint does not read it'
measured = S[S.rises_measured.notna()] if 'rises_measured' in S.columns else S.iloc[:0]
summary = dict(projects=len(S), scored_projects=len(scored), bradyzoite_rank=rank, verdict=verdict,
               rises_in_fingerprint=len(rises), falls_in_fingerprint=len(falls),
               rises_measured_max=(int(measured.rises_measured.max()) if len(measured) else None),
               falls_measured_max=(int(measured.falls_measured.max()) if len(measured) else None),
               minimum_required=MIN_GENES)
json.dump(summary, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print(summary)
measured[['project', 'genes', 'rises_measured', 'falls_measured']].sort_values('genes', ascending=False)

{'projects': 15, 'scored_projects': 0, 'bradyzoite_rank': None, 'verdict': 'no project measures enough of the fingerprint: the depth of these proteomes, not the fingerprint, is what blocks the axis', 'rises_in_fingerprint': 150, 'falls_in_fingerprint': 150, 'rises_measured_max': 10, 'falls_measured_max': 61, 'minimum_required': 40}


,project,genes,rises_measured,falls_measured
2,PXD001042,1653,6,52
14,PXD077563,1558,1,61
12,PXD059833,1510,5,52
8,PXD040598,1278,3,54
9,PXD040635,1256,5,55
0,PXD058095,789,10,36
3,PXD004895,632,0,47
13,PXD064338,306,5,17
10,PXD044588,196,0,27
1,PXD000261,13,0,1


## 3 · Why it fails, which is the finding

The genes that **rise** on conversion are the bradyzoite-specific ones, and they are the ones these proteomes do not reach: at most 10 of 150 in any project, against 17-61 of the 150 that fall. The falling set is ordinary tachyzoite protein, abundant and detected. So the obstacle is depth on the bradyzoite side, not the fingerprint, and a deeper bradyzoite proteome -- not another tachyzoite one -- is what the axis needs.